In [ ]:
# create prices.csv
import requests
import pandas as pd
import os

API_KEY = os.getenv("POLYGON_API_KEY", "")
BASE_URL = "https://api.polygon.io/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{from_}/{to}"

def fetch_aggregates(ticker, start, end, multiplier=1, timespan="day"):
    url = BASE_URL.format(
        ticker=ticker,
        multiplier=multiplier,
        timespan=timespan,
        from_=start,
        to=end
    )
    params = {
        "adjusted": True,
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()  # raises if the status code is not 200
    data = resp.json()
    return data.get("results", [])

def aggs_to_dataframe(records):
    df = pd.DataFrame([{
        "date": pd.to_datetime(rec["t"], unit="ms").date(),
        "open": rec["o"],
        "high": rec["h"],
        "low":  rec["l"],
        "close":rec["c"],
        "volume":rec["v"]
    } for rec in records])
    df.set_index("date", inplace=True)
    return df

if __name__ == "__main__":
    # (1) fetch data
    recs = fetch_aggregates("VT", "2020-01-01", "2025-07-25")
    # (2) convert to DataFrame
    df = aggs_to_dataframe(recs)
    # (3) save as CSV
    df.to_csv("prices.csv")
    print(f"Saved {len(df)} rows to prices.csv")

In [ ]:
# create prices.csv
import requests
import pandas as pd
import os

API_KEY = os.getenv("POLYGON_API_KEY", "")
BASE_URL = "https://api.polygon.io/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{from_}/{to}"

def fetch_aggregates(ticker, start, end, multiplier=1, timespan="day"):
    url = BASE_URL.format(
        ticker=ticker,
        multiplier=multiplier,
        timespan=timespan,
        from_=start,
        to=end
    )
    params = {
        "adjusted": True,
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()  # raises if the status code is not 200
    data = resp.json()
    return data.get("results", [])

def aggs_to_dataframe(records):
    df = pd.DataFrame([{
        "date": pd.to_datetime(rec["t"], unit="ms").date(),
        "open": rec["o"],
        "high": rec["h"],
        "low":  rec["l"],
        "close":rec["c"],
        "volume":rec["v"]
    } for rec in records])
    df.set_index("date", inplace=True)
    return df

if __name__ == "__main__":
    # (1) fetch data
    recs = fetch_aggregates("VT", "2020-01-01", "2025-07-25")
    # (2) convert to DataFrame
    df = aggs_to_dataframe(recs)
    # (3) save as CSV
    df.to_csv("prices.csv")
    print(f"Saved {len(df)} rows to prices.csv")

# Sharpe, Return, and Volatility
import pandas as pd
from pypfopt import expected_returns, risk_models, EfficientFrontier, plotting

# 1. Prepare data: df holds closing prices for several tickers
df = pd.read_csv("prices.csv", parse_dates=True, index_col="date")

# 2. Compute expected returns and the risk model
mu = expected_returns.mean_historical_return(df)          # historical mean return
S  = risk_models.sample_cov(df)                           # sample covariance

# 3. Mean-variance optimisation: maximise the Sharpe ratio
ef = EfficientFrontier(mu, S)
weights = ef.max_sharpe()                                  # optimal weights
cleaned_weights = ef.clean_weights()                       # drop weights below the threshold
print("Optimal Weights:\n", cleaned_weights)

# 4. Report performance metrics
performance = ef.portfolio_performance(verbose=True)
# for example:
# Expected annual return: 12.5%
# Annual volatility: 9.8%
# Sharpe Ratio: 1.27

# 5. Plot the efficient frontier
ef_for_plot = EfficientFrontier(mu, S, solver='SCS')
ax = plotting.plot_efficient_frontier(ef_for_plot, show_assets=False)


In [ ]:
import os
import json
import pandas as pd
import requests
import ast
import time
from tqdm import tqdm
from openai import OpenAI
from pypfopt import expected_returns, risk_models, EfficientFrontier
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

# === Global configuration ===
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")  # read the OpenAI key from the environment
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY", "")  # read the Polygon key from the environment
BASE_URL = (
    "https://api.polygon.io/v2/aggs/ticker/{ticker}/range/"
    "{multiplier}/{timespan}/{from_}/{to}"
)

# === Progress printing helpers ===
def print_step(step, total_steps, description):
    print(f"[{step}/{total_steps}] {description}")

# === 1. Call OpenAI to generate portfolio weights ===
def generate_portfolios():
    client = OpenAI(api_key=OPENAI_API_KEY)
    prompt = f"""
You are an expert portfolio construction advisor.
Based on the past year of US stock performance, generate 3 portfolio suggestions:
1. max_sharpe
2. medium (balanced)
3. min_volatility

Output must be a pure JSON list, each element containing 'name' and 'weights' mappings.
DO NOT include any explanation or extra text.
"""
    resp = client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": prompt}]
    )
    raw = resp.choices[0].message.content
    txt_path = os.path.join("D:/my-fin-project/txt_save", "portfolios.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(raw)
    print("Raw portfolio saved to txt_save/portfolios.txt")
    return clean_portfolios_txt(txt_path)

# === 2. Clean the GPT output and save JSON ===
def clean_portfolios_txt(txt_file, json_file=None):
    with open(txt_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    # Remove potential wrapping lines
    if len(lines) > 2:
        lines = lines[1:-1]
    content = "".join(lines).strip()
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        data = ast.literal_eval(content)
    # Save cleaned JSON
    if not json_file:
        json_file = os.path.join("D:/my-fin-project/json_save", "portfolios.json")
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Cleaned portfolios saved to json_save/{os.path.basename(json_file)}")
    return data

# === 3. Extract weight dicts for the three risk levels ===
def load_portfolios(json_file=None):
    if not json_file:
        json_file = os.path.join("D:/my-fin-project/json_save", "portfolios.json")
    data = json.load(open(json_file, "r", encoding="utf-8"))
    high = next((p["weights"] for p in data if 'max_sharpe' in p['name'].lower() or 'high' in p['name'].lower()), {})
    medium = next((p["weights"] for p in data if 'medium' in p['name'].lower() or 'balanced' in p['name'].lower()), {})
    low = next((p["weights"] for p in data if 'min_volatility' in p['name'].lower() or 'low' in p['name'].lower()), {})
    return high, medium, low

# === 4. Fetch prices and save a wide CSV ===
def fetch_and_save(tickers, start, end, filename, multiplier=1, timespan="day"):  
    records = []
    for t in tqdm(tickers, desc="Fetching data"):
        url = BASE_URL.format(ticker=t, multiplier=multiplier, timespan=timespan, from_=start, to=end)
        params = {"adjusted": True, "sort": "asc", "limit": 50000, "apiKey": POLYGON_API_KEY}
        resp = requests.get(url, params=params); resp.raise_for_status()
        for item in resp.json().get("results", []):
            records.append({
                "date": pd.to_datetime(item["t"], unit="ms"),
                "ticker": t,
                "close": item["c"]
            })
    df = pd.DataFrame(records).pivot(index="date", columns="ticker", values="close")
    path = os.path.join("D:/my-fin-project/csv_save", filename)
    df.to_csv(path)
    print(f"Price data saved to csv_save/{filename}")
    return path

# === 5. Performance analysis and discrete allocation ===
def analyze_performance(csv_file, weights, total_value):
    df = pd.read_csv(csv_file, parse_dates=True, index_col="date")
    mu = expected_returns.mean_historical_return(df)
    S = risk_models.sample_cov(df)
    ef = EfficientFrontier(mu, S)
    # apply custom weights and compute performance
    ef.weights = weights
    ret, vol, sharpe = ef.portfolio_performance(verbose=True)
    latest = get_latest_prices(df)
    da = DiscreteAllocation(weights, latest, total_portfolio_value=total_value)
    alloc, leftover = da.greedy_portfolio()
    print(f"Discrete allocation: {alloc}")
    print(f"Funds remaining: ${leftover:.2f}")
    return ret, vol, sharpe, alloc, leftover

# === 6. Main pipeline ===
def run_pipeline(start="2024-07-25", end="2025-07-25", total_value=10000):
    steps = 2 + 3 * 3
    s = 1; t0 = time.time()
    print_step(s, steps, "Generating portfolio weights via AI"); portfolios = generate_portfolios(); s+=1
    print_step(s, steps, "Loading portfolio JSON"); high, med, low = load_portfolios(); s+=1
    results = []
    for level, w in [("high", high), ("medium", med), ("low", low)]:
        print_step(s, steps, f"Fetching prices for the {level} portfolio"); csvp = fetch_and_save(list(w.keys()), start, end, f"{level}_portfolio.csv"); s+=1
        print_step(s, steps, f"Analysing performance of the {level} portfolio"); perf = analyze_performance(csvp, weights=w, total_value=total_value); s+=1
        print_step(s, steps, f"Saving results for the {level} portfolio");
        ret, vol, sharpe, alloc, _ = perf
        out = {"risk_level": level, "tickers": list(w.keys()), "allocation": alloc,
               "exp_return": ret, "volatility": vol, "sharpe": sharpe}
        outf = os.path.join("D:/my-fin-project/output_save", f"{level}_performance.json")
        with open(outf, "w", encoding="utf-8") as f: json.dump(out, f, ensure_ascii=False, indent=2)
        print(f"Results saved to output_save/{level}_performance.json")
        results.append(out)
        s+=1
    print(f"Pipeline complete: {time.time()-t0:.2f}s")
    return results

# === Entry point ===
if __name__ == "__main__":
    all_res = run_pipeline()
    print("All portfolio results:", all_res)


In [ ]:
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

df = pd.read_csv("D:\my-fin-project\csv_save\high_portfolio.csv", parse_dates=["date"], index_col="date")
latest_prices = get_latest_prices(df)

print("Index name:", df.index.name)
print("Columns:", df.columns.tolist())

In [ ]:
latest_prices = get_latest_prices(df)
print("Latest prices dict:", latest_prices)

In [ ]:
print("Weights keys:", weights.keys())
# expected: dict_keys(['AAPL','AMZN','MSFT','NVDA','TSLA'])


In [ ]:
da = DiscreteAllocation(weights, latest_prices, total_portfolio_value=10000)
allocation, leftover = da.greedy_portfolio()
print("Discrete allocation:", allocation)
print("Funds remaining: ${:.2f}".format(leftover))